# QAP.ia — Inventário privado de gravações

## tl;dr

O banco ativo contém **68 reuniões** e **19,65 horas** de áudio referenciado. A triagem encontrou **30 reuniões elegíveis para anotação** e selecionou uma amostra estratificada de **18 reuniões / 9,98 horas**. A amostra ainda não é um gold set: transcrições e atas precisam de revisão humana antes de qualquer treino.

## Context & Methods

A unidade de análise é uma reunião ativa no SwiftData. Apenas segmentos referenciados pelo banco são inspecionados. O notebook não lê títulos nem participantes e não grava conteúdo de transcrição ou resumo em seus outputs.

### Key Assumptions

- O proprietário do ambiente autorizou o uso privado destas gravações para preparação de amostra.
- Consentimento, base legal e agrupamento por participante/organização permanecem gates anteriores ao treino.
- Atas existentes são rótulos ruidosos; não são tratadas como referência humana.
- Pastas sem registro correspondente no banco são consideradas órfãs e ficam fora da seleção.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

search_roots = [Path.cwd(), *Path.cwd().parents]
repo_root = next(path for path in search_roots if (path / 'AI').is_dir())
script_path = repo_root / 'AI/scripts/profile_recorded_audio.py'
output_root = repo_root / 'AI/runs/recorded-audio-inventory'
private_map = repo_root / 'AI/data/recorded-audio-private-map.json'

completed = subprocess.run(
    [
        sys.executable, str(script_path),
        '--hash-audio',
        '--sample-size', '18',
        '--output-root', str(output_root),
        '--private-map', str(private_map),
    ],
    cwd=repo_root,
    capture_output=True,
    text=True,
    check=True,
)
summary = json.loads(completed.stdout)
summary

{'schema_version': 1,
 'grain': 'one active SwiftData meeting',
 'privacy': {'titles_read': False,
  'participants_read': False,
  'text_content_written_to_reports': False,
  'raw_audio_copied': False},
 'inventory': {'database_meetings': 68,
  'database_states': {'completed': 58, 'failed': 10},
  'filesystem_meeting_directories': 400,
  'database_meetings_missing_directory': 0,
  'orphan_filesystem': {'directories': 332,
   'with_audio': 9,
   'with_transcript': 252,
   'with_summary': 250},
  'referenced_audio_files': 68,
  'decodable_referenced_audio_files': 68,
  'referenced_audio_hours': 19.65,
  'duplicate_audio_groups': 0},
 'quality': {'eligible_for_annotation': 30,
  'ineligible': 38,
  'failure_counts': {'audio_shorter_than_5_minutes': 32,
   'transcript_too_short': 31,
   'summary_too_short': 22,
   'implausible_transcript_density': 13,
   'meeting_not_completed': 10,
   'no_referenced_audio': 7}},
 'sample': {'requested_meetings': 18,
  'selected_meetings': 18,
  'selected_

## Data

O perfil abaixo contém somente contagens, duração e flags de integridade. A relação entre `sample_id` e o arquivo original fica em `AI/data/`, diretório ignorado pelo Git e protegido com permissão de leitura exclusiva do usuário.

In [2]:
inventory = summary['inventory']
quality = summary['quality']
sample = summary['sample']

overview = {
    'Reuniões ativas': inventory['database_meetings'],
    'Pastas no filesystem': inventory['filesystem_meeting_directories'],
    'Pastas órfãs': inventory['orphan_filesystem']['directories'],
    'Áudios referenciados': inventory['referenced_audio_files'],
    'Áudios decodificáveis': inventory['decodable_referenced_audio_files'],
    'Horas referenciadas': inventory['referenced_audio_hours'],
    'Elegíveis para anotação': quality['eligible_for_annotation'],
    'Selecionadas': sample['selected_meetings'],
    'Horas selecionadas': sample['selected_audio_hours'],
}
overview

{'Reuniões ativas': 68,
 'Pastas no filesystem': 400,
 'Pastas órfãs': 332,
 'Áudios referenciados': 68,
 'Áudios decodificáveis': 68,
 'Horas referenciadas': 19.65,
 'Elegíveis para anotação': 30,
 'Selecionadas': 18,
 'Horas selecionadas': 9.98}

## Results

A principal perda de cobertura vem de reuniões muito curtas, transcrições pequenas e atas insuficientes. Esses registros podem ser testes, gravações interrompidas ou reuniões sem conteúdo bastante para supervisão.

In [3]:
sorted(
    summary['quality']['failure_counts'].items(),
    key=lambda item: (-item[1], item[0]),
)

[('audio_shorter_than_5_minutes', 32),
 ('transcript_too_short', 31),
 ('summary_too_short', 22),
 ('implausible_transcript_density', 13),
 ('meeting_not_completed', 10),
 ('no_referenced_audio', 7)]

In [4]:
{
    'Amostra por duração': sample['duration_buckets'],
    'Status': sample['status'],
    'Split': sample['split'],
}

{'Amostra por duração': {'long_45_plus_min': 5,
  'medium_20_to_45_min': 8,
  'short_5_to_20_min': 5},
 'Status': 'annotation_pool_not_training_ready',
 'Split': 'pending_participant_and_organization_grouping'}

## Takeaways

1. As 18 reuniões selecionadas cobrem faixas curtas, médias e longas e formam um **pool de anotação**, não um conjunto pronto para treino.
2. As 332 pastas órfãs ficam excluídas porque não têm vínculo confiável com o estado atual do produto; nove delas ainda contêm áudio e devem ser revisadas separadamente, sem exclusão automática.
3. Antes do envio ao servidor, o pool precisa passar por autorização/consentimento, agrupamento por participantes e organizações e revisão humana de transcrição e ata.
4. O split final deve manter todas as gravações da mesma reunião, organização e participantes em uma única partição para evitar vazamento.